In [8]:
# 03_deep_dive.ipynb
# Deep dive analysis: Earnings Quality, YoY Change, Company Snapshot
# Requires: 01_data_collection.ipynb to have been run first

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pickle
import numpy as np

# Define companies — must match 01_data_collection.ipynb
tickers = {
    "Tesla": "TSLA",
    "BYD": "BYDDY",
    "Ford": "F"
}
# Load data
with open('../data/metrics.pkl', 'rb') as f:
    metrics = pickle.load(f)

with open('../data/raw_data.pkl', 'rb') as f:
    data = pickle.load(f)

# Consistent colors
colors = {
    "Tesla": "#E31937",
    "BYD": "#1DB954",
    "Ford": "#003DA5"
}

print("Data loaded successfully")
for name in metrics:
    print(f"{name}: {metrics[name].index.year.tolist()}")

Data loaded successfully
Tesla: [2022, 2023, 2024]
BYD: [2022, 2023, 2024]
Ford: [2022, 2023, 2024]


In [9]:
# Section 1: Year-over-Year Change Analysis

# Calculate YoY % change for each metric
yoy = {}

for name in metrics:
    yoy[name] = metrics[name].diff()
    yoy[name] = yoy[name].dropna()  # First year will be NaN

# Build grouped bar chart for each metric
metrics_to_plot = [
    ("Net Profit Margin (%)", "Net Profit Margin"),
    ("Free Cash Flow", "Free Cash Flow"),
    ("Return on Assets (%)", "Return on Assets")
]

fig_yoy = make_subplots(
    rows=1, cols=3,
    subplot_titles=[m[1] for m in metrics_to_plot]
)

years = ["2022 -> 2023", "2023 -> 2024"]

for col, (metric, title) in enumerate(metrics_to_plot, start=1):
    for name in yoy:
        fig_yoy.add_trace(go.Bar(
            x=years,
            y=yoy[name][metric].values,
            name=name,
            marker_color=colors[name],
            legendgroup=name,
            showlegend=(col == 1)
        ), row=1, col=col)

fig_yoy.update_layout(
    title_text="Year-over-Year Absolute Change (%) — Tesla vs BYD vs Ford",
    template="plotly_white",
    height=450,
    width=1200,
    barmode="group",
    hovermode="x unified"
)

fig_yoy.show()

In [11]:
# Section 2: Earnings Quality — Cash Conversion Ratio
# Cash Conversion Ratio = Operating Cash Flow / Net Income
# Ratio < 1 means company is reporting more profit than it's actually generating in cash
# Red flag if consistently low or declining

earnings_quality = {}

for name in tickers.keys():
    operating_cf = data[name]["cashflow"].loc["Operating Cash Flow"]
    net_income = data[name]["income"].loc["Net Income"]
    
    ccr = operating_cf / net_income
    
    earnings_quality[name] = pd.DataFrame({
        "Cash Conversion Ratio": ccr
    }).sort_index()

    earnings_quality[name] = earnings_quality[name].dropna()
    earnings_quality[name] = earnings_quality[name][
    earnings_quality[name].index.year <= current_year
]
    
    from datetime import datetime
    current_year = datetime.today().year - 2
    earnings_quality[name] = earnings_quality[name][
        earnings_quality[name].index.year <= current_year
    ]

print("Cash Conversion Ratios:")
for name in earnings_quality:
    print(f"\n{name}:")
    print(earnings_quality[name])

Cash Conversion Ratios:

Tesla:
            Cash Conversion Ratio
2022-12-31               1.170150
2023-12-31               0.883792
2024-12-31               2.092987

BYD:
            Cash Conversion Ratio
2022-12-31               8.472739
2023-12-31               5.649815
2024-12-31               3.315266

Ford:
            Cash Conversion Ratio
2022-12-31              -3.459364
2023-12-31               3.431792
2024-12-31               2.623405


In [13]:
fig_ccr = go.Figure()

for name in earnings_quality:
    fig_ccr.add_trace(go.Scatter(
        x=earnings_quality[name].index.year,
        y=earnings_quality[name]["Cash Conversion Ratio"],
        name=name,
        mode="lines+markers",
        line=dict(color=colors[name], width=2),
        marker=dict(size=8)
    ))

# Add reference line at 1.0 — below this means profit > cash generated
fig_ccr.add_hline(
    y=1.0,
    line_dash="dash",
    line_color="grey",
    annotation_text="CCR = 1.0 (breakeven)",
    annotation_position="bottom right"
)

fig_ccr.update_layout(
    title="Earnings Quality: Cash Conversion Ratio — Tesla vs BYD vs Ford (2022–2024)",
    xaxis_title="Year",
    yaxis_title="Cash Conversion Ratio",
    legend_title="Company",
    hovermode="x unified",
    template="plotly_white"
)

fig_ccr.update_xaxes(
    tickmode="array",
    tickvals=[2022, 2023, 2024],
    ticktext=["2022", "2023", "2024"]
)

fig_ccr.show()

Tesla hugs the 1.0 line the whole period — cash generation barely keeping pace with reported profits. Dipped below in 2023 which is the red flag moment.\
BYD well above 1.0 but declining steeply — still healthy but the trend is the concern. At this rate it converges with Tesla and Ford by 2026. \
Ford negative in 2022 due to the net loss distortion, then jumps above 1.0 and stabilizes — actually the most consistent cash converter in 2023-2024 despite lowest margins. \

That last point is counterintuitive and analytically interesting — Ford's earnings quality is arguably better than Tesla's right now despite Tesla having much higher margins. That's the kind of insight that distinguishes real analysis from surface-level chart reading.